# TensorRT 800×960 · GWC Plugin V3 Single Engine · 40 组批处理

这个 notebook 在反序列化前加载并注册 `libGWCVolumePlugin.so`，然后只加载一次 `onnx/foundationstereo_800x960_gwc_plugin.engine`，依次处理 `data/` 中完整的 40 组双目数据。每组均在固定 **800×960** TensorRT 网格上推理、投影和建 mesh（不对 mesh 下采样）；相机内参同步缩放，因此 3D 坐标与面积仍是现实尺度。

输出不会覆盖 PyTorch 的 `result/`：

- `result/tensorrt_800x960_gwc_plugin/<group>/<capture>/comparison.png`：TensorRT 输入、mask、右图和 TensorRT disparity；
- `result/tensorrt_800x960_gwc_plugin/<group>/<capture>/mesh.ply`：800×960 网格构建的三角 mesh；
- `result/tensorrt_800x960_gwc_plugin/areas.json`：40 组 TensorRT mesh 面积，单位 m²。

不保存 TIFF、NPY 或独立 point cloud。若任一组失败，仍会完成其他组，但不会写最终 `areas.json`。

## 1. 环境、40 组输入与独立输出目录

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from time import perf_counter
import ctypes
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorrt as trt
import torch

from fs_high_resolution_utils import (
    build_constrained_mesh,
    load_rectify_and_select_mask,
    postprocess_disparity_gpu,
    save_triangle_mesh,
)

REPO_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'FoundationStereo').is_dir() and (candidate / 'onnx').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from the FS_Engine repository or python/ directory.')
REPO_ROOT = REPO_ROOT.resolve()

DATA_ROOT = REPO_ROOT / 'data'
OUTPUT_ROOT = REPO_ROOT / 'result' / 'tensorrt_800x960_gwc_plugin'
ENGINE_PATH = REPO_ROOT / 'onnx' / 'foundationstereo_800x960_gwc_plugin.engine'
PLUGIN_LIBRARY_PATH = REPO_ROOT / 'cpp' / 'build' / 'libGWCVolumePlugin.so'
MODEL_HEIGHT, MODEL_WIDTH = 800, 960
ORIGINAL_WIDTH, ORIGINAL_HEIGHT = 2448, 2048
DEVICE = torch.device('cuda:0')
REQUIRED_INPUTS = ('left.png', 'right.png', 'calibration.json', 'masks/left_mask.png')

if not torch.cuda.is_available():
    raise RuntimeError('PyTorch CUDA is required.')
if not ENGINE_PATH.is_file():
    raise FileNotFoundError(f'Engine not found: {ENGINE_PATH}')
if not PLUGIN_LIBRARY_PATH.is_file():
    raise FileNotFoundError(f'Plugin library not found: {PLUGIN_LIBRARY_PATH}. Run VS Code Build TSFS first.')

def load_and_register_gwc_plugin(plugin_path: Path):
    # Keep this CDLL object alive for the whole notebook: TensorRT needs the
    # Plugin V3 implementation while deserializing and executing the engine.
    library = ctypes.CDLL(str(plugin_path), mode=ctypes.RTLD_GLOBAL)
    register = library.fs_register_gwc_volume_plugin
    register.argtypes = []
    register.restype = ctypes.c_bool
    if not register():
        raise RuntimeError(f'Failed to register GWCVolume Plugin V3 from {plugin_path}')
    return library

GWC_PLUGIN_LIBRARY = load_and_register_gwc_plugin(PLUGIN_LIBRARY_PATH)

def capture_sort_key(path):
    try:
        return (0, int(path.name))
    except ValueError:
        return (1, path.name)

CAPTURES = sorted(
    (capture_dir
     for group_dir in sorted(DATA_ROOT.iterdir()) if group_dir.is_dir()
     for capture_dir in sorted(group_dir.iterdir(), key=capture_sort_key)
     if capture_dir.is_dir() and all((capture_dir / item).is_file() for item in REQUIRED_INPUTS)),
    key=lambda path: (path.parent.name, capture_sort_key(path)),
)
if len(CAPTURES) != 40:
    raise RuntimeError(f'Expected 40 complete captures in {DATA_ROOT}, found {len(CAPTURES)}.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for capture_dir in CAPTURES:
    (OUTPUT_ROOT / capture_dir.relative_to(DATA_ROOT)).mkdir(parents=True, exist_ok=True)

print('Repository:', REPO_ROOT)
print('Engine:', ENGINE_PATH)
print('Plugin:', PLUGIN_LIBRARY_PATH)
print('TensorRT:', trt.__version__)
print('GPU:', torch.cuda.get_device_name(DEVICE))
print(f'Found {len(CAPTURES)} captures; TensorRT outputs: {OUTPUT_ROOT}')


Repository: /home/liu4000/Desktop/FS_Engine
Engine: /home/liu4000/Desktop/FS_Engine/onnx/foundationstereo_800x960_gwc_plugin.engine
Plugin: /home/liu4000/Desktop/FS_Engine/cpp/build/libGWCVolumePlugin.so
TensorRT: 11.2.1.2
GPU: NVIDIA RTX PRO 4000 Blackwell
Found 40 captures; TensorRT outputs: /home/liu4000/Desktop/FS_Engine/result/tensorrt_800x960_gwc_plugin


## 2. 只加载一次 TensorRT engine

In [2]:
class FoundationStereoTensorRT:
    INPUT_NAMES = ('left', 'right')
    OUTPUT_NAME = 'disp'

    def __init__(self, engine_path: Path, height: int, width: int):
        self.height, self.width = height, width
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)
        self.engine = self.runtime.deserialize_cuda_engine(engine_path.read_bytes())
        if self.engine is None:
            raise RuntimeError(f'Could not deserialize {engine_path}')
        self.context = self.engine.create_execution_context()
        if self.context is None:
            raise RuntimeError('Could not create TensorRT execution context')
        self.stream = torch.cuda.Stream(device=DEVICE)
        self.tensor_names = tuple(self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors))
        if set(self.tensor_names) != {*self.INPUT_NAMES, self.OUTPUT_NAME}:
            raise RuntimeError(f'Unexpected engine I/O: {self.tensor_names}')
        input_shape = (1, 3, height, width)
        for name in self.INPUT_NAMES:
            if not self.context.set_input_shape(name, input_shape):
                raise RuntimeError(f'Unable to set {name} shape to {input_shape}')
        if tuple(self.context.get_tensor_shape(self.OUTPUT_NAME)) != (1, 1, height, width):
            raise RuntimeError('Unexpected disparity output shape')

    def _to_device_tensor(self, image_rgb: np.ndarray) -> torch.Tensor:
        resized = cv2.resize(image_rgb, (self.width, self.height), interpolation=cv2.INTER_LINEAR)
        nchw = np.ascontiguousarray(resized.transpose(2, 0, 1)[None].astype(np.float32))
        return torch.from_numpy(nchw).to(DEVICE, non_blocking=True)

    def infer(self, left_rgb: np.ndarray, right_rgb: np.ndarray) -> torch.Tensor:
        # Use one non-default stream for upload and TensorRT enqueue.
        with torch.cuda.stream(self.stream):
            left = self._to_device_tensor(left_rgb)
            right = self._to_device_tensor(right_rgb)
            disparity = torch.empty((1, 1, self.height, self.width), device=DEVICE, dtype=torch.float32)
            self.context.set_tensor_address('left', left.data_ptr())
            self.context.set_tensor_address('right', right.data_ptr())
            self.context.set_tensor_address('disp', disparity.data_ptr())
            if not self.context.execute_async_v3(self.stream.cuda_stream):
                raise RuntimeError('TensorRT enqueue failed')
        return disparity

runner = FoundationStereoTensorRT(ENGINE_PATH, MODEL_HEIGHT, MODEL_WIDTH)
print('Engine I/O:', runner.tensor_names)
print(f'Fixed input/output grid: {MODEL_WIDTH}×{MODEL_HEIGHT}')


Engine I/O: ('left', 'right', 'disp')
Fixed input/output grid: 960×800


## 3. 批量推理、真实比例投影、导出 comparison / PLY / areas.json

In [3]:
Z_FAR_M = 1.0
REMOVE_INVISIBLE, DENOISE_CLOUD = True, True
DENOISE_MAX_NEIGHBOR_DISTANCE_M = 0.01
DENOISE_MIN_NEIGHBORS, DENOISE_EDGE_MIN_NEIGHBORS = 3, 2
DOWNSAMPLE_PIXELS = 1  # Build mesh from every retained point on the 800×960 grid.
MESH_MAX_EDGE_M, MESH_MAX_DEPTH_JUMP_M = 0.02, 0.01

def scaled_model_intrinsics(rectified_k: np.ndarray, source_width: int, source_height: int) -> np.ndarray:
    # Disparity is measured in model-grid pixels, so fx/fy/cx/cy must use that same grid.
    scale_x, scale_y = source_width / MODEL_WIDTH, source_height / MODEL_HEIGHT
    k_model = rectified_k.copy().astype(np.float32)
    k_model[0, :] /= scale_x
    k_model[1, :] /= scale_y
    k_model[2, :] = (0.0, 0.0, 1.0)
    return k_model

def save_tensorrt_comparison(left_rgb, right_rgb, mask, disparity, output_path, capture_label):
    mask_overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    mask_overlay[mask] = (0.0, 1.0, 0.0, 0.45)
    finite = disparity[np.isfinite(disparity) & (disparity > 0)]
    vmax = float(np.percentile(finite, 99)) if finite.size else 1.0

    figure, axes = plt.subplots(1, 4, figsize=(25, 6), constrained_layout=True)
    panels = (
        (left_rgb, 'TensorRT input · left · 960×800'),
        (left_rgb, 'TensorRT input · left + mask'),
        (right_rgb, 'TensorRT input · right · 960×800'),
        (disparity, 'TensorRT disparity · 960×800'),
    )
    for axis, (image, title) in zip(axes, panels):
        if image.ndim == 2:
            rendered = axis.imshow(image, cmap='turbo', vmin=0, vmax=vmax)
        else:
            axis.imshow(image)
        axis.set_title(title)
        axis.axis('off')
    axes[1].imshow(mask_overlay, interpolation='none')
    figure.colorbar(rendered, ax=axes[3], shrink=0.8, label='disparity (model-grid pixels)')
    figure.suptitle(capture_label)
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(figure)

# Warm up engine once; this result is intentionally discarded.
first_capture = CAPTURES[0]
warmup_left = cv2.cvtColor(cv2.imread(str(first_capture / 'left.png')), cv2.COLOR_BGR2RGB)
warmup_right = cv2.cvtColor(cv2.imread(str(first_capture / 'right.png')), cv2.COLOR_BGR2RGB)
warmup_output = runner.infer(warmup_left, warmup_right)
runner.stream.synchronize()
del warmup_left, warmup_right, warmup_output
torch.cuda.empty_cache()

areas_m2 = {}
failures = []
total_inference_seconds = 0.0

for number, capture_dir in enumerate(CAPTURES, start=1):
    relative_path = capture_dir.relative_to(DATA_ROOT)
    group_name, capture_name = relative_path.parts
    output_dir = OUTPUT_ROOT / relative_path
    left_path = capture_dir / 'left.png'
    right_path = capture_dir / 'right.png'
    mask_path = capture_dir / 'masks' / 'left_mask.png'
    calibration_path = capture_dir / 'calibration.json'

    try:
        rectified_left, rectified_right, rectified_k, baseline_m, rectified_left_mask = (
            load_rectify_and_select_mask(
                globals(), left_path, right_path, mask_path, calibration_path,
                ORIGINAL_WIDTH, ORIGINAL_HEIGHT,
            )
        )
        source_height, source_width = rectified_left.shape[:2]
        if (source_width, source_height) != (ORIGINAL_WIDTH, ORIGINAL_HEIGHT):
            raise RuntimeError(f'Unexpected rectified size: {source_width}×{source_height}')

        started = torch.cuda.Event(enable_timing=True)
        finished = torch.cuda.Event(enable_timing=True)
        with torch.cuda.stream(runner.stream):
            started.record(runner.stream)
            disparity_cuda = runner.infer(rectified_left, rectified_right)
            finished.record(runner.stream)
        finished.synchronize()
        inference_ms = started.elapsed_time(finished)
        total_inference_seconds += inference_ms / 1000.0
        disparity = np.ascontiguousarray(disparity_cuda.squeeze().detach().cpu().numpy(), dtype=np.float32)
        if disparity.shape != (MODEL_HEIGHT, MODEL_WIDTH) or not np.isfinite(disparity).all():
            raise RuntimeError(f'Invalid TensorRT disparity: shape={disparity.shape}')

        model_left = cv2.resize(rectified_left, (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_LINEAR)
        model_right = cv2.resize(rectified_right, (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_LINEAR)
        model_mask = cv2.resize(
            rectified_left_mask.astype(np.uint8), (MODEL_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_NEAREST
        ).astype(bool)
        k_model = scaled_model_intrinsics(rectified_k, source_width, source_height)

        surface = postprocess_disparity_gpu(
            torch.from_numpy(disparity).to(DEVICE, non_blocking=True),
            k_model, baseline_m, model_mask, DEVICE, Z_FAR_M,
            remove_invisible=REMOVE_INVISIBLE,
            denoise=DENOISE_CLOUD,
            max_neighbor_distance_m=DENOISE_MAX_NEIGHBOR_DISTANCE_M,
            min_neighbors=DENOISE_MIN_NEIGHBORS,
            edge_min_neighbors=DENOISE_EDGE_MIN_NEIGHBORS,
        )
        mesh_result = build_constrained_mesh(
            surface.xyz_map, surface.keep_mask, model_mask, model_left,
            downsample_pixels=DOWNSAMPLE_PIXELS,
            max_edge_m=MESH_MAX_EDGE_M,
            max_depth_jump_m=MESH_MAX_DEPTH_JUMP_M,
        )
        save_triangle_mesh(mesh_result, output_dir / 'mesh.ply')
        save_tensorrt_comparison(
            model_left, model_right, model_mask, disparity, output_dir / 'comparison.png', str(relative_path)
        )
        areas_m2.setdefault(group_name, {})[capture_name] = mesh_result.area_m2
        print(
            f'[{number:02d}/{len(CAPTURES)}] {relative_path}: '
            f'{mesh_result.area_m2:.6f} m² ({mesh_result.area_m2 * 1e4:.2f} cm²), {inference_ms:.1f} ms'
        )
    except Exception as error:
        failures.append((str(relative_path), f'{type(error).__name__}: {error}'))
        print(f'[{number:02d}/{len(CAPTURES)}] FAILED {relative_path}: {error}')
    finally:
        for variable_name in (
            'rectified_left', 'rectified_right', 'rectified_k', 'baseline_m', 'rectified_left_mask',
            'disparity_cuda', 'disparity', 'model_left', 'model_right', 'model_mask', 'k_model',
            'surface', 'mesh_result',
        ):
            globals().pop(variable_name, None)
        torch.cuda.empty_cache()

if failures:
    formatted = '\n'.join(f' - {path}: {message}' for path, message in failures)
    raise RuntimeError(f'{len(failures)} capture(s) failed; areas.json was not written:\n{formatted}')

areas_path = OUTPUT_ROOT / 'areas.json'
areas_payload = {
    'unit': 'm²',
    'engine': str(ENGINE_PATH.relative_to(REPO_ROOT)),
    'input_grid': [MODEL_HEIGHT, MODEL_WIDTH],
    'mesh_downsample_pixels': DOWNSAMPLE_PIXELS,
    'areas_m2': areas_m2,
}
areas_path.write_text(json.dumps(areas_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Completed {len(CAPTURES)} captures in {total_inference_seconds:.1f} s of TensorRT inference.')
print(f'Areas JSON: {areas_path}')


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[01/40] Volunteer2_lower/0: 0.018688 m² (186.88 cm²), 531.0 ms
[02/40] Volunteer2_lower/1: 0.019255 m² (192.55 cm²), 529.5 ms
[03/40] Volunteer2_lower/2: 0.020033 m² (200.33 cm²), 525.3 ms
[04/40] Volunteer2_lower/3: 0.019624 m² (196.24 cm²), 529.5 ms
[05/40] Volunteer2_lower/4: 0.019717 m² (197.17 cm²), 526.0 ms
[06/40] Volunteer2_lower/5: 0.020283 m² (202.83 cm²), 530.4 ms
[07/40] Volunteer2_lower/6: 0.019381 m² (193.81 cm²), 530.1 ms
[08/40] Volunteer2_lower/7: 0.019607 m² (196.07 cm²), 532.5 ms
[09/40] Volunteer2_lower/8: 0.020172 m² (201.72 cm²), 532.4 ms
[10/40] Volunteer2_lower/9: 0.019979 m² (199.79 cm²), 533.4 ms
[11/40] Volunteer2_upper/0: 0.087917 m² (879.17 cm²), 533.2 ms
[12/40] Volunteer2_upper/1: 0.087406 m² (874.06 cm²), 532.5 ms
[13/40] Volunteer2_upper/2: 0.088942 m² (889.42 cm²), 529.7 ms
[14/40] Volu

## 输出位置

成功完成后，40 组结果都在 `result/tensorrt_800x960_attention_axis_fixed/` 下；`areas.json` 的目录结构与 `data/<group>/<capture>/` 一致。